[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/expand_dataset.ipynb)

# Adaptive Data: Expand a Dataset

Take 500 rows of [GSM8K](https://huggingface.co/datasets/openai/gsm8k) from Hugging Face, then grow them three ways - **domain**, **general** and **similar** rows, 100 of each and inspect what each kind of row looks like.

**Docs:** [/adaptive-data/expand-data/](https://docs.adaptionlabs.ai/adaptive-data/expand-data/)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

> ⏱ Runtime: ~1 hour · 💳 Cost: low (3 × 100 expansion rows; the exact figure is printed before the run)

In [1]:
%pip install -q adaption pandas pyarrow

/Users/andrijazz/source/adaption-runbooks/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

Everything a reader might change lives here, so the rest of the notebook runs unedited.

In [4]:
PARQUET_URL = "https://huggingface.co/datasets/openai/gsm8k/resolve/main/main/train-00000-of-00001.parquet"
COLUMN_MAPPING = {"prompt": "question", "completion": "answer"}

SEED_ROWS = 500  # rows of GSM8K to upload; this is the dataset we expand

# One expansion per strategy, so each output dataset holds the seed plus one kind of row.
EXPANSIONS = {
    "domain": {"domain_rows": 100},
    "general": {"general_rows": 100},
    "similar": {"similar_rows": 100},
}

SAMPLES_TO_SHOW = 3

In [5]:
import html

import pandas as pd
from IPython.display import HTML, Markdown, display


def wait(dataset_id: str, what: str):
    """Wait for a dataset job and raise with the API's message if it failed."""
    status = client.datasets.wait_for_completion(dataset_id, timeout=3 * 3600)
    if status.status == "failed":
        msg = (status.error_data and status.error_data.message) or "unknown error"
        raise RuntimeError(f"{what} failed: {msg}")
    print(f"{what}: {status.status}, {status.row_count} rows")
    return status


def download(dataset_id: str, path: str) -> pd.DataFrame:
    client.datasets.download(dataset_id, file_format="jsonl").write_to_file(path)
    return pd.read_json(path, lines=True)


def first_present(row, columns):
    for col in columns:
        value = row.get(col)
        if isinstance(value, str) and value.strip() and value.strip() != "N/A":
            return value
    return ""


PROMPT_COLS = ["original_prompt", "enhanced_prompt", "prompt"]
COMPLETION_COLS = ["original_completion", "enhanced_completion", "completion"]


def show_rows(df: pd.DataFrame, title: str, n: int = SAMPLES_TO_SHOW, seed: int = 0):
    """Render n rows as prompt/completion cards."""
    sample = df.sample(min(n, len(df)), random_state=seed)
    cards = []
    for _, row in sample.iterrows():
        tag = row.get("augmented_approach")
        badge = (
            f"<span style='font-size:11px;padding:2px 8px;border-radius:10px;"
            f"background:#e0e7ff;color:#3730a3'>{html.escape(str(tag))}</span>"
            if isinstance(tag, str)
            else ""
        )
        prompt = html.escape(first_present(row, PROMPT_COLS))
        completion = html.escape(first_present(row, COMPLETION_COLS))
        cards.append(
            "<div style='border:1px solid #d4d4d8;border-radius:8px;padding:12px 14px;margin:10px 0'>"
            f"<div style='margin-bottom:6px'>{badge}</div>"
            f"<div style='font-weight:600;margin-bottom:4px'>Prompt</div>"
            f"<div style='white-space:pre-wrap;margin-bottom:10px'>{prompt}</div>"
            f"<div style='font-weight:600;margin-bottom:4px'>Completion</div>"
            f"<div style='white-space:pre-wrap;color:#52525b'>{completion}</div>"
            "</div>"
        )
    display(HTML(f"<h4>{html.escape(title)}</h4>" + "".join(cards)))


def added_rows(df: pd.DataFrame, seed_prompts: set) -> pd.DataFrame:
    """Rows the expansion added, as opposed to the seed rows it carries over."""
    if "augmented_approach" in df.columns and df["augmented_approach"].notna().any():
        return df[df["augmented_approach"].notna()]
    if "source" in df.columns and (df["source"] == "augmented_data").any():
        return df[df["source"] == "augmented_data"]
    # Fallback: anything whose prompt is not one of the seed prompts.
    prompts = df.apply(lambda r: first_present(r, PROMPT_COLS), axis=1)
    return df[~prompts.isin(seed_prompts)]

## Step 1 - Take a 500-row slice of GSM8K

GSM8K is ~7.5k grade-school math word problems, each with a step-by-step solution. We download the train split from Hugging Face and keep a random 500 rows.

In [24]:
import urllib.request

urllib.request.urlretrieve(PARQUET_URL, "gsm8k-train.parquet")
seed_df = pd.read_parquet("gsm8k-train.parquet").sample(SEED_ROWS, random_state=0)
seed_df.to_parquet("gsm8k-seed.parquet", index=False)
print(f"{len(seed_df)} rows, columns: {list(seed_df.columns)}")

500 rows, columns: ['question', 'answer']


## Step 2 - Upload

Expansion needs a dataset that is ready to read. Instead of running Adaptive Data on it, we upload the rows with `processing_mode="raw"`, which takes them as-is: `question` becomes the prompt and `answer` the completion. See [`datasets.upload_file`](https://docs.adaptionlabs.ai/api/python/resources/datasets/methods/create).

In [8]:
upload = client.datasets.upload_file(
    "gsm8k-seed.parquet",
    name="GSM8K seed (500 rows)",
    processing_mode="raw",
    column_mapping=COLUMN_MAPPING,
)
dataset_id = upload.dataset_id
print(dataset_id)

wait(dataset_id, "Upload")

b3a31b54-075e-4355-b81a-471cb66b0c78
Upload: succeeded, 500 rows


DatasetGetStatusResponse(dataset_id='b3a31b54-075e-4355-b81a-471cb66b0c78', status='succeeded', row_count=500, progress=Progress(percent=100, processed_rows=500, total_rows=500), error_data=None)

In [9]:
seed_view = seed_df.rename(columns={"question": "prompt", "answer": "completion"})
seed_prompts = set(seed_view["prompt"])
show_rows(seed_view, "Seed rows (GSM8K, as uploaded)")

## Step 3 - Expand: domain, general and similar rows

[`datasets.augment`](https://docs.adaptionlabs.ai/api/python/resources/datasets/methods/augment) returns a **new dataset** holding the seed rows plus the added ones; the seed is left unchanged. It supports three kinds of row:

| Strategy | Parameter | Where rows come from |
| --- | --- | --- |
| Domain | `domain_rows` | Retrieved from Adaption's curated pool, **matching the topics** of your dataset |
| General | `general_rows` | Retrieved from the curated pool, from **other topics**, to broaden coverage |
| Similar | `similar_rows` | **Generated** fresh, in the style, language and format of your rows |

The three can be combined in a single call. Here we launch one expansion per strategy so each output contains just one kind of added row and we can compare them side by side. First, price all three:

In [10]:
total = 0
for name, rows in EXPANSIONS.items():
    est = client.datasets.augment(dataset_id, **rows, estimate=True)
    total += est.estimated_credits_consumed
    print(f"{name:>8}: {est.estimated_credits_consumed} credits")
print(f"{'total':>8}: {total} credits")

  domain: 1.0 credits
 general: 1.0 credits
 similar: 1.0 credits
   total: 3.0 credits


Launch all three. The `idempotency_key` makes re-running this cell safe: a repeat returns the dataset already created instead of starting (and charging for) a second run.

In [11]:
expanded_ids = {}
for name, rows in EXPANSIONS.items():
    response = client.datasets.augment(
        dataset_id,
        **rows,
        idempotency_key=f"expand-runbook-{dataset_id}-{name}",
    )
    expanded_ids[name] = response.dataset_id
    print(f"{name:>8}: {response.dataset_id} ({response.status})")

  domain: 764b3e20-8194-4fb0-95a8-ab80b2409d86 (running)
 general: 1d6db731-5f96-4b27-9951-6b75a8f2dd98 (running)
 similar: 8a2b1229-0eb9-4604-a308-d234ac69a8b7 (running)


Wait for each expansion. A timeout only stops waiting, not the job — re-run the cell to resume.

In [13]:
for name, expanded_id in expanded_ids.items():
    wait(expanded_id, f"{name} expansion")

domain expansion: succeeded, 600 rows
general expansion: succeeded, 600 rows
similar expansion: succeeded, 600 rows


In [14]:
expanded = {}
for name, expanded_id in expanded_ids.items():
    df = download(expanded_id, f"gsm8k-{name}.jsonl")
    new = added_rows(df, seed_prompts)
    expanded[name] = new
    print(f"{name:>8}: {len(df)} rows total = {len(df) - len(new)} seed + {len(new)} added")

if "augmented_approach" in expanded["domain"].columns:
    display(
        pd.concat(expanded.values())["augmented_approach"]
        .value_counts()
        .rename_axis("augmented_approach")
        .to_frame("added rows")
    )

  domain: 600 rows total = 500 seed + 100 added
 general: 600 rows total = 500 seed + 100 added
 similar: 600 rows total = 500 seed + 100 added


## Step 4 - Inspect each kind of row

Added rows are tagged with `augmented_approach` (`domain-specific`, `domain-diverse` or `similar`) and `source = "augmented_data"`; seed rows leave both empty, so you can always tell which rows you bought.

### Domain rows - more of the same topic

Adaption first works out the topic mix of your dataset - for GSM8K, math, arithmetic and multi-step word problems. It then **retrieves** real rows on those topics from its curated pool. Nothing is generated: these are existing, quality-filtered examples, so they bring new problems and new phrasings while staying on-topic.

Use domain rows when your dataset is too small to train on but the pool already covers your domain.

In [20]:
show_rows(expanded["domain"], "Domain rows")

### General rows - broader coverage

General rows are also **retrieved** from the curated pool, but deliberately from topics *outside* your dataset's mix - coding, writing, science, everyday questions and so on. They are not meant to teach the target task. They keep a fine-tuned model from over-specialising and losing its general abilities (catastrophic forgetting).

A common recipe is to keep roughly 40% general rows alongside your domain data.

In [21]:
show_rows(expanded["general"], "General rows")

### Similar rows - generated in your style

Similar rows are **generated**, not retrieved. Adaption reads a sample of your rows, works out their topics, language and prompt/completion conventions - here: a short word problem, a worked solution with `<<…>>` calculator annotations, and a final `#### <answer>` line — and writes new rows that follow them.

Use similar rows when the pool has little that looks like your data, or when the format itself matters. You are billed only for the rows actually generated, up to `similar_rows`.

In [22]:
show_rows(expanded["similar"], "Similar rows")


## Next steps

- Combine the seed and the expansions you want into one training set with [`datasets.combine`](https://docs.adaptionlabs.ai/adaptive-data/combine-datasets/), or request all three kinds in one `augment` call.
- Add other languages with [`translate` and `localize`](https://docs.adaptionlabs.ai/adaptive-data/expand-data/#translate-rows).
- Train on the result with the [AutoScientist runbook](../autoscientist/autoscientist.ipynb).